In [17]:
CSV_PATH="maize_data.csv"; FAST_DEBUG=True; RANDOM_SEED=42
TEST_SIZE=0.2; CV_FOLDS=5; N_JOBS=-1
MAX_SFS_FEATS=25 if not FAST_DEBUG else 10
CANDIDATE_POOL=200 if not FAST_DEBUG else 100

import re, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.feature_selection import SequentialFeatureSelector, f_regression
from sklearn.base import clone
rng=np.random.RandomState(RANDOM_SEED)

rmse=lambda yt,yp: float(np.sqrt(mean_squared_error(yt,yp)))
rmse_scorer=make_scorer(lambda yt,yp: np.sqrt(mean_squared_error(yt,yp)),greater_is_better=False)

def timed_gridsearch(pipe,grid,Xtr,ytr,Xte,yte,name,cv=CV_FOLDS):
    t=time.perf_counter()
    gs=GridSearchCV(pipe,grid,scoring=rmse_scorer,cv=cv,n_jobs=N_JOBS,refit=True,error_score="raise").fit(Xtr,ytr)
    yhat=gs.best_estimator_.predict(Xte)
    return {"Method":name,"RMSE_CV":-float(gs.best_score_),"RMSE_Test":rmse(yte,yhat),
            "MAE_Test":mean_absolute_error(yte,yhat),"Time_s":time.perf_counter()-t,
            "Best_Params":gs.best_params_,"Best_Model":gs.best_estimator_}

count_nonzero=lambda pl,X,y: int(np.sum(np.asarray(clone(pl).fit(X,y).named_steps["model"].coef_).ravel()!=0))

def stability(best_pl,X,y,repeats=5,ts=0.2):
    s=pd.Series(0,index=X.columns,dtype=int)
    for _ in range(repeats):
        Xtr,_,ytr,_=train_test_split(X,y,test_size=ts,random_state=int(rng.randint(1e6)))
        coefs=np.asarray(clone(best_pl).fit(Xtr,ytr).named_steps["model"].coef_).ravel()
        s+=(coefs!=0).astype(int)
    return (s/repeats).sort_values(ascending=False)

def _sniff(sample):
    first=sample.splitlines()[0] if sample else ""
    sep=max([';',',','\t','|'],key=first.count) if first else ';'
    dec=',' if re.search(r'\d,\d',sample) and not re.search(r'\d\.\d',sample) else '.'
    return sep,dec

def read_csv_robust(p: str|Path):
    p=Path(p)
    try: sample=p.open('rb').read(131072).decode('utf-8','ignore')
    except: sample=""
    sep,dec=_sniff(sample)
    for enc in ('utf-8','latin-1'):
        try:
            return pd.read_csv(p,sep=sep,decimal=dec,engine="python",encoding=enc,
                               on_bad_lines="skip",na_values=["","NA","NaN"],
                               quotechar='"',doublequote=True,escapechar='\\',thousands=' ')
        except: pass
    return pd.read_csv(p,sep=';',decimal=',',engine="python",encoding='latin-1',
                       on_bad_lines="skip",na_values=["","NA","NaN"],
                       quotechar='"',doublequote=True,escapechar='\\',thousands=' ')

df=read_csv_robust(CSV_PATH)
print(f"[INFO] rows={len(df):,}, cols={len(df.columns):,}")

cl = {c.lower(): c for c in df.columns}
yname = cl.get("dtoa") or cl.get("dtoa_") or cl.get("dtoa_days") or ("DtoA" if "DtoA" in df.columns else None)
assert yname in df.columns, f"response not found; first cols: {list(df.columns)[:12]}"
snps = [c for c in df.columns if re.fullmatch(r"m\d+", str(c))]
assert snps, "no m1..mN"

X = df[snps].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(df[yname], errors="coerce")

keep = y.notna()
X, y = X.loc[keep], y.loc[keep]

X = X.loc[:, X.notna().any(axis=0)]
row_keep = X.notna().sum(axis=1) >= max(5, int(0.01 * X.shape[1]))
X, y = X.loc[row_keep], y.loc[row_keep]

if FAST_DEBUG:
    X = X.sample(n=min(1200, len(X)), random_state=RANDOM_SEED)
    y = y.loc[X.index]
    if X.shape[1] > 1500:
        X = X.iloc[:, :1500]

pre = ColumnTransformer(
    [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                       ("sc", StandardScaler())]), list(X.columns))],
    remainder="drop"
)

Xtr, Xte, ytr, yte = train_test_split(X, y.values, test_size=TEST_SIZE, random_state=RANDOM_SEED)
print(f"[READY] Xtr={Xtr.shape}, Xte={Xte.shape}")

# models
results=[]; QUICK=True; cv=3 if QUICK else CV_FOLDS

# Ridge
ridge=Pipeline([("prep",pre),("model",Ridge(max_iter=10000))])
r_out=timed_gridsearch(ridge,{"model__alpha":np.logspace(-1,2,4) if QUICK else np.logspace(-2,3,10)},Xtr,ytr,Xte,yte,"Ridge",cv)
results.append({k:v for k,v in r_out.items() if k!="Best_Model"}); ridge_best=r_out["Best_Model"]

# Lasso
lasso=Pipeline([("prep",pre),("model",Lasso(max_iter=5000,tol=1e-3))])
l_out=timed_gridsearch(lasso,{"model__alpha":np.logspace(-2,0.7,5) if QUICK else np.logspace(-3,1,10)},Xtr,ytr,Xte,yte,"Lasso",cv)
results.append({k:v for k,v in l_out.items() if k!="Best_Model"}); lasso_best=l_out["Best_Model"]; lasso_nz=count_nonzero(lasso_best,Xtr,ytr)

# Elastic Net
enet=Pipeline([("prep",pre),("model",ElasticNet(max_iter=5000,tol=1e-3))])
e_grid={"model__alpha":np.logspace(-2,0.7,5),"model__l1_ratio":[0.3,0.7]} if QUICK else {"model__alpha":np.logspace(-3,1,8),"model__l1_ratio":[0.2,0.5,0.8]}
e_out=timed_gridsearch(enet,e_grid,Xtr,ytr,Xte,yte,"Elastic Net",cv)
results.append({k:v for k,v in e_out.items() if k!="Best_Model"}); enet_best=e_out["Best_Model"]; enet_nz=count_nonzero(enet_best,Xtr,ytr)

# Stepwise (SFS)
t=time.perf_counter()
prep_only=Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())])
Xtr_a=prep_only.fit_transform(Xtr); Xte_a=prep_only.transform(Xte)
f,_=f_regression(Xtr_a,ytr); pool=np.argsort(f)[::-1][:min(CANDIDATE_POOL,Xtr_a.shape[1])]
Xtr_p,Xte_p=Xtr_a[:,pool],Xte_a[:,pool]
sfs=SequentialFeatureSelector(LinearRegression(),n_features_to_select=min(MAX_SFS_FEATS,Xtr_p.shape[1]),
                              direction="forward",scoring=make_scorer(lambda yt,yp:-rmse(yt,yp),greater_is_better=True),
                              cv=CV_FOLDS,n_jobs=N_JOBS).fit(Xtr_p,ytr)
sel=np.where(sfs.get_support())[0]
yhat_sfs=LinearRegression().fit(Xtr_p[:,sel],ytr).predict(Xte_p[:,sel])
results.append({"Method":f"SFS ({len(sel)} feats)","RMSE_CV":np.nan,"RMSE_Test":rmse(yte,yhat_sfs),
               "MAE_Test":mean_absolute_error(yte,yhat_sfs),"Time_s":time.perf_counter()-t,
               "Best_Params":{"candidate_pool":int(Xtr_p.shape[1]),"n_selected":int(len(sel))}})

# PCR / PLS
pcr=Pipeline([("prep",pre),("pca",PCA(svd_solver="full")),("lr",LinearRegression())])
pls=Pipeline([("prep",pre),("pls",PLSRegression())])
pcr_out=timed_gridsearch(pcr,{"pca__n_components":[10,25,50] if QUICK else [10,25,50,100,200]},Xtr,ytr,Xte,yte,"PCR")
pls_out=timed_gridsearch(pls,{"pls__n_components":[2,5,10] if QUICK else [2,5,10,20,40]},Xtr,ytr,Xte,yte,"PLS")
results.append({k:v for k,v in pcr_out.items() if k!="Best_Model"})
results.append({k:v for k,v in pls_out.items() if k!="Best_Model"})

# comparison table + figures + residuals + stability
res=pd.DataFrame(results)
print(res[["Method","RMSE_CV","RMSE_Test","MAE_Test","Time_s","Best_Params"]])

# Bar chart of Test RMSE
plt.figure(); plt.bar(res["Method"],res["RMSE_Test"]); plt.ylabel("RMSE (Test)")
plt.title("Method Comparison — RMSE(Test)"); plt.xticks(rotation=45,ha="right")
plt.tight_layout(); plt.savefig("rmse_bar.png",dpi=180); plt.close()
print("[Saved] rmse_bar.png")

# RMSE vs Time
plt.figure(); plt.scatter(res["Time_s"],res["RMSE_Test"])
for i,m in enumerate(res["Method"]):
    plt.annotate(m,(res["Time_s"].iloc[i],res["RMSE_Test"].iloc[i]),fontsize=8)
plt.xlabel("Time (s)"); plt.ylabel("RMSE (Test)"); plt.title("RMSE vs Compute Time")
plt.tight_layout(); plt.savefig("rmse_vs_time.png",dpi=180); plt.close()
print("[Saved] rmse_vs_time.png")

# Save results table with Feats/Comps
def feats_or_comps(row):
    if row["Method"].startswith("Lasso"):   return int(lasso_nz)
    if row["Method"].startswith("Elastic"): return int(enet_nz)
    if row["Method"].startswith("PCR"):     return int(pcr_out["Best_Model"].named_steps["pca"].n_components_)
    if row["Method"].startswith("PLS"):     return int(pls_out["Best_Model"].named_steps["pls"].n_components)
    if "SFS" in row["Method"]:              return int(row["Best_Params"]["n_selected"])
    return int(Xtr.shape[1])
res["Feats_or_Comps"]=res.apply(feats_or_comps,axis=1)
res[["Method","RMSE_CV","RMSE_Test","MAE_Test","Feats_or_Comps","Time_s","Best_Params"]].to_csv("results_table.csv",index=False)
print("[Saved] results_table.csv")

# Residuals plot for BEST model
models_map={
    "Ridge": ridge_best, "Lasso": lasso_best, "Elastic Net": enet_best,
    "PCR": pcr_out["Best_Model"], "PLS": pls_out["Best_Model"]
}
best_row = res.loc[res["RMSE_Test"].idxmin(),"Method"]
# align on prefix to map
for key in models_map:
    if best_row.startswith(key):
        pick = key; break
if pick is None:
    pick = "Elastic Net"
best_pipe = models_map[pick]
yhat = best_pipe.predict(Xte); resid = yte - yhat
plt.figure(figsize=(5,4))
plt.scatter(yhat, resid, alpha=0.6)
plt.axhline(0, color="red", ls="--"); plt.xlabel("Predicted DtoA"); plt.ylabel("Residuals")
plt.title(f"Residuals vs Predicted — {pick}")
plt.tight_layout(); plt.savefig("residuals_plot.png", dpi=180); plt.close()
print("[Saved] residuals_plot.png")

# Stability summaries for sparse models
lasso_stab = stability(lasso_best, X, y, repeats=5)
enet_stab  = stability(enet_best,  X, y, repeats=5)
lasso_stab.head(10).to_csv("lasso_stability_top10.txt", sep="\t")
enet_stab.head(10).to_csv("enet_stability_top10.txt",  sep="\t")
print("[Saved] lasso_stability_top10.txt, enet_stability_top10.txt")


[INFO] rows=4,981, cols=7,394
[READY] Xtr=(960, 1500), Xte=(240, 1500)
           Method   RMSE_CV  RMSE_Test  MAE_Test     Time_s  \
0           Ridge  3.885541   3.868705  3.076698   2.426663   
1           Lasso  3.605314   3.723680  2.943533   6.675874   
2     Elastic Net  3.605344   3.740668  2.944645  17.429221   
3  SFS (10 feats)       NaN   3.768474  3.016985  29.326063   
4             PCR  3.595421   3.756290  2.992640  15.062003   
5             PLS  3.605631   3.689693  2.958470   5.798628   

                                         Best_Params  
0                            {'model__alpha': 100.0}  
1                {'model__alpha': 0.223872113856834}  
2  {'model__alpha': 1.0592537251772898, 'model__l...  
3          {'candidate_pool': 100, 'n_selected': 10}  
4                          {'pca__n_components': 25}  
5                           {'pls__n_components': 2}  
[Saved] rmse_bar.png
[Saved] rmse_vs_time.png
[Saved] results_table.csv
[Saved] residuals_plot.png
[Sa